### Query Translation - Multi Query
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some
way as to improve retrival.

Semantic search on embeddings is hard to get right. Embedding long documents is
especially challenging. User queries are a challenge, if user provides an ambigious
query, they'll get an ambiguous matches and hence an ambguous answer (because we are doing semantic similarity searches) ! LLMs just follow what was in the context and hallucinate answers as a result.

One approach is to take the query & re-write it (or reframing it) from a different perspective. Multi-query is one such approach, another is RAG Fusion. In multi query, we break a larger user query into multiple queries; for each query we find the matching contexts, which we can combine later into a larger context for LLM to answer from. The intuition is that this improves the search results.

![Multi Query](images/multi_query.png)

In [3]:
import bs4, sys, os
import pathlib
from dotenv import load_dotenv
from typing import List, TypedDict
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END

In [2]:
# load API keys from .env files
load_dotenv(override=True)
# for colorful text output
console = Console()

In [6]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
faiss_store = pathlib.Path(os.getcwd()) / "faiss_index_rag_mq"

In [7]:
pathlib.Path(os.getcwd())

WindowsPath('c:/Dev/Code/git-projects/learning_langchain/src/langchain_tutorial')

In [10]:
def create_or_load_embeddings():
    """creates if not available or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=300, chunk_overlap=50
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        # Use a Gemini embedding model that is suitable for retrieval.
        # It is important to match the model to the task.
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vectorstore.as_retriever()
        vectorstore.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )

    return retriever

In [11]:
retriever = create_or_load_embeddings()

Loading document from URL https://lilianweng.github.io/posts/2023-06-23-agent/. Please wait...

Loaded 1 documents from URL

Metadata of first document: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}

First 200 chars of first document: 

      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a 

Chunking the PDF. Please wait...

Created 50 chunks

Creating embeddings. Please wait...

Local embeddings created at c:\Dev\Code\git-projects\learning_langchain\src\langchain_tutorial\faiss_index_rag_mq

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives
    # Swap out ChatOpenAI for the Gemini 2.5 Flash model
    | ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [14]:
generate_queries = (
    prompt_perspectives | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke({"question": "What is task decomposition for LLM agents?"})

['How do large language model agents break down complex tasks into smaller, manageable sub-tasks?',
 'What are the benefits and methods of task decomposition for improving LLM agent performance?',
 'Explain the concept of hierarchical planning and sub-goal generation in the context of LLM-based agents.',
 'Describe the process by which an LLM agent performs task breakdown to achieve a complex objective.',
 'What strategies do autonomous LLM agents use to modularize problems or divide a main goal?']

In [15]:
from langchain.load import dumps, loads


def get_unique_union(documents: list[list]):
    """Unique union of retrieved docs"""
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

In [16]:
# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question": question})
print(f"Got {len(docs)} documents")

Got 8 documents


C:\Users\manis\AppData\Local\Temp\ipykernel_19204\3928205432.py:11: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


In [17]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

final_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

response = final_rag_chain.invoke({"question": question})
console.print(response)

For LLM agents, **task decomposition** is the process of breaking down large, complicated tasks into smaller, more 
manageable subgoals or steps. This enables the agent to handle complex tasks efficiently and plan ahead.

Key aspects of task decomposition include:
*   **Enhancing performance:** Techniques like Chain of Thought (CoT) prompting instruct the model to "think step 
by step," transforming big tasks into multiple simpler ones.
*   **Exploring possibilities:** Tree of Thoughts (ToT) extends CoT by decomposing problems into multiple thought 
steps and generating multiple reasoning possibilities per step, forming a tree structure.
*   **Methods of decomposition:** This can be achieved by the LLM itself using simple prompts (e.g., "Steps for 
XYZ"), through task-specific instructions, or with human input.